# Step 10 · Train a reward model

A score head on the base model learns score(chosen) > score(rejected) (TRL RewardTrainer + LoRA).

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
from lab import jobs, reward_model
cfg = dict(jobs.RM_PRESETS["Tiny (smoke test)"])
rid = jobs.start("rm", BASE, latest_ds("pref"), None, cfg, wait=True)
run = registry.run(rid); RM = run["output_model_id"]; print(run["status"], "→", RM, run["error"] or "")

In [ ]:
ref = reward_model.info(RM)
print(f"held-out accuracy {ref['val_accuracy']:.0%} (train {ref['train_accuracy']:.0%}) on {ref['n_val']} pairs — 50% = guessing")
pd.DataFrame(ref["pairs"])

In [ ]:
# Score any two answers
scorer = reward_model.Scorer(RM)
q = "Can I promise a customer their lawn mower arrives tomorrow?"
scorer.score([q, q], ["I will check warehouse stock and delivery slots before confirming a date.",
                      "Yes, it will definitely arrive tomorrow."])

In [ ]:
del scorer   # free GPU memory before RLHF training (notebook 08)